# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided as a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
# Access metadata properties
print("Dataset Name: {}\nDescription: {}".format(metadata.name, metadata.description))

## 2. Data Overview
Review available record sets, fields, and their `@id`s as described by the Croissant schema.

In [ ]:
# Print overview of record sets and their fields using @id
record_sets = metadata.recordSet
if not record_sets:
    # Try loading and printing available record sets from dataset
    print("No record sets explicitly listed in metadata. Fetching from dataset...")
    ds_record_sets = dataset.record_sets
    for rs in ds_record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        if 'field' in rs:
            fields = rs['field']
            for f in fields:
                print(f"  Field @id: {f['@id']} ({f.get('name','')})")
else:
    for rs in record_sets:
        print("RecordSet @id: {}".format(rs['@id']))
        if 'field' in rs:
            for f in rs['field']:
                print("  Field @id: {} ({})".format(f['@id'], f.get('name','')))

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s identified above.

In [ ]:
# Fetch all available RecordSet @id's from the dataset
record_sets_info = dataset.record_sets
record_set_ids = [rs['@id'] for rs in record_sets_info]
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Loaded record set: {record_set_id} with shape {dataframes[record_set_id].shape}")
        else:
            print(f"Record set {record_set_id} returned no records.")
    except Exception as e:
        print(f"Could not load records for record set {record_set_id}: {e}")

# Print columns for the first DataFrame loaded
if dataframes:
    first_rs = list(dataframes.keys())[0]
    print(f"Available columns for record set {first_rs}:")
    print(dataframes[first_rs].columns.tolist())
    dataframes[first_rs].head()
else:
    print("No DataFrames loaded from record sets.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations include removing outliers, transforming data distributions, or grouping data by key attributes.

In [ ]:
# EDA on a selected record set
if dataframes:
    df = list(dataframes.values())[0]
    rs_id = list(dataframes.keys())[0]
    print(f"Using RecordSet @id: {rs_id}")

    # Try to pick a numeric field by guessing from columns
    numeric_fields = [col for col in df.columns if df[col].dtype in [float, int]]
    if not numeric_fields:
        # Try to cast columns to numeric
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce')
            except Exception:
                continue
        numeric_fields = [col for col in df.columns if df[col].dtype in [float, int]]

    if numeric_fields:
        numeric_field = numeric_fields[0]
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())

        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, norm_col]].head())

        # Pick a group field (categorical)
        group_fields = [col for col in df.columns if df[col].dtype == object]
        if group_fields:
            group_field = group_fields[0]
            if group_field in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
                print(f"Grouped data by {group_field}:")
                print(grouped_df.head())
            else:
                print(f"Group field {group_field} not in filtered dataframe.")
        else:
            print("No categorical/group fields found.")
    else:
        print("No numeric fields found for EDA.")
else:
    print("No dataframes available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Modify or extend based on the available fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize numeric field distribution if available
if dataframes:
    df = list(dataframes.values())[0]
    numeric_fields = [col for col in df.columns if df[col].dtype in [float, int]]
    if not numeric_fields:
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce')
            except Exception:
                continue
        numeric_fields = [col for col in df.columns if df[col].dtype in [float, int]]
    if numeric_fields:
        field = numeric_fields[0]
        plt.figure(figsize=(6,4))
        sns.histplot(df[field], bins=10, kde=True)
        plt.title(f"Distribution of {field}")
        plt.xlabel(field)
        plt.ylabel('Count')
        plt.show()
    else:
        print("No numeric fields found for visualization.")
else:
    print("No dataframe available for visualization.")

## 6. Conclusion
This notebook has demonstrated the process of loading and exploring a FAIR^2 dataset using `mlcroissant`, referencing all entities by their `@id` for reproducibility and traceability.

**Key findings:**
- The dataset includes structured tables on clinical, hormone, imaging, and genetic features for three patients.
- It highlights how to access and analyze its record sets and fields using their unique `@id` identifiers.
- Basic filtering, normalization, grouping, and visualization can be accomplished using standard tools in combination with `mlcroissant`.

Further analysis can adapt this workflow to more advanced processing or statistical comparisons, always referencing data elements by their `@id` according to the Croissant standard.